# Train an autoencoder

STL-10 has **13,000 labelled images** (5,000 train + 8,000 test) and 100,000 unlabelled
ones. This notebook uses the 13,000. The autoencoder learns
from the images alone.

The network squeezes a 96 x 96 colour image into a **256-dimensional** vector and tries to
rebuild the image from it. That vector is the feature we use in `06_`.

Output: `data/autoencoder_stl10.pth` and `data/ae_embeddings_stl10_1000.npz`.

## All labelled images

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset

DATA = "data/stl10_binary"
IMG = 96
CH = 3

def load_rgb(split):
    X = np.fromfile(f"{DATA}/{split}_X.bin", dtype=np.uint8).reshape(-1, CH, IMG, IMG)
    return X.transpose(0, 3, 2, 1)

X_all = np.concatenate([load_rgb("train"), load_rgb("test")])

print(X_all.shape, X_all.dtype)
print(f"{X_all.nbytes / 1e6:.0f} MB in memory")

### Train / validation split

No labels are involved, so this is just a random split.

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(X_all))

#n_val = len(X_all) // 2
#val_idx = order[:n_val]
#train_idx = order[n_val:]
val_idx = order[:2500]
train_idx = order[2500:5000]

print(f"train {len(train_idx):,} / val {len(val_idx):,}")

In [ ]:
class Images(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]).permute(2, 0, 1).float().div(255)

train_loader = DataLoader(Images(X_all[train_idx]), batch_size=200, shuffle=True, num_workers=0)
val_loader = DataLoader(Images(X_all[val_idx]), batch_size=200, shuffle=False, num_workers=0)

print(len(train_loader), len(val_loader), "batches")

## 1. Autoencoder architecture

- Encoder: Conv(32, 64, 128, 256) with stride 2 → 6 x 6 x 256 → Linear to 256
- Decoder: Linear back to 6 x 6 x 256 → ConvTranspose up to 96 x 96 x 3 → sigmoid

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 256 * 6 * 6), nn.ReLU(),
            nn.Unflatten(1, (256, 6, 6)),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)

model = Autoencoder().to(device)

print(device, f"| {sum(p.numel() for p in model.parameters()):,} parameters")

x = torch.randn(1, 3, IMG, IMG).to(device)
print(f"input       {tuple(x.shape)}")
print(f"latent      {tuple(model.encoder(x).shape)}")
print(f"reconstruct {tuple(model(x).shape)}")

## 2. Train

Loss: mean squared error between the image and its reconstruction.

In [ ]:
EPOCHS = 10
LR = 1e-3

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train": [], "val": []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for x in train_loader:
        x = x.to(device)
        loss = criterion(model(x), x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(x)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device)
            val_loss += criterion(model(x), x).item() * len(x)
    val_loss /= len(val_loader.dataset)

    history["train"].append(train_loss)
    history["val"].append(val_loss)
    print(f"epoch {epoch + 1:3}/{EPOCHS}  train {train_loss:.5f}  val {val_loss:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

ax.plot(history["train"], label="train")
ax.plot(history["val"], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "data/autoencoder_stl10.pth")
print("saved data/autoencoder_stl10.pth")

## 3. Reconstruction

In [ ]:
model.eval()

X_show = X_all[val_idx[:8]]
x = torch.from_numpy(X_show).permute(0, 3, 1, 2).float().div(255).to(device)

with torch.no_grad():
    recon = model(x).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))

for i in range(8):
    axes[0, i].imshow(X_show[i])
    axes[1, i].imshow(recon[i].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].axis("off")
    axes[1, i].axis("off")

axes[0, 0].set_title("original", loc="left", fontsize=9)
axes[1, 0].set_title("reconstruction", loc="left", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Embeddings for the 1,000 images

The same 1,000 images as `03_Extract_Features.ipynb`, taken from `idx`

In [ ]:
d = np.load("data/embeddings_stl10_1000.npz")

idx = d["idx"]
y = d["y"]

X = X_all[idx]

@torch.no_grad()
def embed(images, batch_size=128):
    out = []
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(images[start:start + batch_size])
        batch = batch.permute(0, 3, 1, 2).float().div(255).to(device)
        out.append(model.encoder(batch).cpu().numpy())
        if start % (batch_size * 5) == 0:
            print(f"  {start:,} / {len(images):,}", flush=True)
    return np.vstack(out)

E = embed(X)

np.savez_compressed("data/ae_embeddings_stl10_1000.npz",
                    idx=idx, y=y, E=E.astype(np.float32))

print(f"saved data/ae_embeddings_stl10_1000.npz  E {E.shape}")